# 08 · Databricks SDK, Connect and the UI

**Run first:** `just tour list`, then each step `just tour whoami` … `just tour delete`, looking
at the UI location printed for each step. Code: `docs/learn/live-setup/databricks_tour.py`.

Cells that talk to your workspace run only when `DTRACK_NB_LIVE=1` is set before starting the
kernel (profile `corp` in `~/.databrickscfg`).

In [ ]:
import os

import pandas as pd
from nbkit import table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

LIVE = os.environ.get("DTRACK_NB_LIVE") == "1"
print("live cells:", "on" if LIVE else "off")

## 1 · Two clients, two jobs

| | `WorkspaceClient` (databricks-sdk) | `DatabricksSession` (databricks-connect) |
| --- | --- | --- |
| talks | REST APIs | Spark Connect (gRPC) |
| does | jobs, files, Volumes, Unity Catalog objects, SQL statements | DataFrames and `spark.sql` |
| where the work runs | the API service | remote serverless compute; this PC only sends plans and receives rows |
| used by | `provision.py`, `test_job_flow`, `just tour` | `tests/live/dtrack_right/test_databricks_target.py` |

Connect needs a client version that matches the serverless runtime, which is why `.venv-live`
uses Python 3.12.

## 2 · Three identities

| Identity | Proven by | Does |
| --- | --- | --- |
| you on AWS | `AWS_PROFILE` keys | every `dtrack` write to S3 |
| you on Databricks | profile `corp` (token here, browser login at work) | every SDK call, including `storage_credentials.create` |
| Databricks for the job | IAM role `dtrack-live-uc` | reads S3 behind `/Volumes/...` |

## 3 · Unity Catalog objects

- `__databricks_managed_storage_credential` / `__databricks_managed_storage_location` come with
  the workspace: Databricks' own bucket for **managed** tables. They were not created by
  `provision.py` and cannot be deleted. Only `dtrack_live` (credential + location) is ours.
- `SHOW GRANTS` lists explicit grants only. As owner and admin you hold every privilege
  implicitly, so the list is empty. A colleague needs `GRANT READ VOLUME ... TO ...`.

In [ ]:
if LIVE:
    from databricks.sdk import WorkspaceClient

    w = WorkspaceClient(profile="corp")
    for location in w.external_locations.list():
        print(f"{location.name:42} {location.url[:60]}")
else:
    print("set DTRACK_NB_LIVE=1 to list external locations")

## 4 · Jobs: submit vs create, reset vs update

- `jobs.submit` → a one-time `SUBMIT_RUN`: visible under **Runs**, never under **Jobs**.
  The live tests use it so nothing permanent is left behind.
- `jobs.create` → a persistent job with a schedule: visible under **Jobs**.
- `jobs.reset(job_id, new_settings)` **replaces all settings**. `jobs.update(job_id,
  new_settings)` merges only the fields given.

**Common mistake:** reading `reset` as "restore defaults". It is "overwrite everything", so
`just tour edit` first `get`s the full settings, changes three fields, and resets with the whole
object. Below, the same edit is simulated on plain dicts.

In [ ]:
current = {
    "name": "dtrack-tour",
    "tasks": [{"task_key": "hello", "notebook_path": "/Users/me/dtrack_tour/hello"}],
    "schedule": {"cron": "0 0 6 * * ?", "pause_status": "PAUSED"},
    "tags": {"owner": "dtrack-tour"},
}
change = {"schedule": {"cron": "0 0 6 * * ?", "pause_status": "UNPAUSED"}}

reset_with_partial = dict(change)
update_with_partial = {**current, **change}
reset_with_full = {**current, **change, "tags": {**current["tags"], "edited": "now"}}
table(
    pd.DataFrame(
        [
            {
                "call": label,
                "tasks kept": str("tasks" in settings),
                "schedule": settings["schedule"]["pause_status"],
                "tags": str(settings.get("tags", "-")),
            }
            for label, settings in (
                ("reset(partial)", reset_with_partial),
                ("update(partial)", update_with_partial),
                ("reset(get + edit)", reset_with_full),
            )
        ]
    )
)

## 5 · Audit and history from system tables

`just tour edit` shows up as `reset` in the audit log; delete a job and its runs remain in
history. Run these in the SQL Editor or with the live cell below.

```sql
SELECT event_time, action_name, request_params['job_id'] AS job_id
FROM system.access.audit
WHERE service_name = 'jobs' AND event_date >= current_date() - INTERVAL 1 DAY
ORDER BY event_time DESC;

SELECT period_start_time, job_id, run_id, run_type, result_state
FROM system.lakeflow.job_run_timeline ORDER BY period_start_time DESC;
```

In [ ]:
if LIVE:
    warehouse = next(iter(w.warehouses.list())).id
    result = w.statement_execution.execute_statement(
        warehouse_id=warehouse,
        wait_timeout="50s",
        statement="SELECT action_name, count(*) AS n FROM system.access.audit WHERE service_name = 'jobs' GROUP BY 1 ORDER BY 2 DESC",
    )
    display(pd.DataFrame(result.result.data_array or [], columns=["action_name", "n"]))
else:
    print("set DTRACK_NB_LIVE=1 to query system.access.audit")

## 6 · Safe deletion order

`provision.py teardown` deletes dependents before what they depend on: Volumes → external
locations → storage credentials → secret scopes → schemas → IAM users → roles → buckets, one
confirmation each, and only objects recorded in `.provision.json`. Deleting a credential that a
location still uses fails unless forced; the UI shows the same error.